In [1]:
from faker import Faker
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os

fake = Faker('es_ES')


In [7]:
hoy = datetime.now()
hace_un_anio = hoy - timedelta(days=365)
hace_un_anio

datetime.datetime(2025, 5, 13, 12, 22, 20, 21132)

## Generate customers
- customer_id
- company_name
- industry
- country
- plan_type
- mrr
- contract_start
- contract_end
- churned
- churn_date
- employee_count
- health_score

In [93]:
def generar_cliente():
    rangos_mmr = {
        "Starter": (99, 299),
        "Pro": (300, 999),
        "Enterprise": (1000, 5000)
    }
    plan_type = random.choice(["Starter", "Pro", "Enterprise"])
    churned = fake.boolean()
    contract_start = fake.date_between(start_date="-4y", end_date='today')
    contract_end = contract_start + timedelta(days=30*random.choice([1,3,6,12]))
    if churned == True: 
        churned_date = fake.date_between(start_date=contract_start, end_date=contract_end)
        health_score = random.randint(0,50)
    else:
        churned_date = None
        health_score = random.randint(40,100)


    return {
        "customer_id":      fake.uuid4(),
        "company_name":     fake.company(),
        "industry":         random.choice(["Tech", "Retail", "Finance", "Health"]),
        "country":          fake.country(),
        "plan_type":        plan_type,
        "mrr":              random.randint(*rangos_mmr[plan_type]),
        "contract_start":   contract_start,
        "contract_end":     contract_end,
        "churned":          churned,
        "churned_date":     churned_date,
        "employee_count":   random.randint(10,5000),
        "health_score":     health_score
    }

print(generar_cliente())

{'customer_id': 'c58b51b5-12cf-4a03-a874-1ca056c325ce', 'company_name': 'Armengol & Asociados S.L.L.', 'industry': 'Retail', 'country': 'El Salvador', 'plan_type': 'Pro', 'mrr': 654, 'contract_start': datetime.date(2025, 10, 19), 'contract_end': datetime.date(2026, 4, 17), 'churned': True, 'churned_date': datetime.date(2026, 2, 5), 'employee_count': 666, 'health_score': 32}


In [94]:
customers = [generar_cliente() for _ in range(3000)]

In [96]:
df_customers = pd.DataFrame(data=customers)
df_customers

,customer_id,company_name,industry,country,plan_type,mrr,contract_start,contract_end,churned,churned_date,employee_count,health_score
0,949c8a0e-c8f2-40cc-8f97-527bcd35406a,Construcción Castellana S.Coop.,Health,Iraq,Enterprise,4972,2023-12-19,2024-03-18,True,2024-01-16,108,10
1,bb69a3c1-816c-454b-8d94-481186e8c99a,Castells y asociados S.Com.,Retail,Hungría,Starter,135,2025-07-26,2026-07-21,False,None,1532,91
2,5220a66c-36b8-4037-8c43-b08eb81203e2,Riquelme & Asociados S.Com.,Finance,Túnez,Enterprise,4468,2024-02-23,2024-03-24,False,None,909,97
3,b6878b37-e5da-460d-9d8b-aafa4eb5b029,Alberto y asociados S.Com.,Health,Argelia,Pro,846,2022-08-13,2023-08-08,False,None,3559,93
4,3439ad77-7bbb-41c9-88f8-1f276cf29283,Úrsula Lopez Morante S.Coop.,Retail,Niger,Pro,848,2023-01-31,2023-07-30,False,None,4096,86
...,...,...,...,...,...,...,...,...,...,...,...,...
2995,24d727ee-2ce0-4db9-a64f-e82458f8a86f,Aranda y Becerra S.L.,Health,Venezuela,Enterprise,1832,2023-01-17,2023-07-16,True,2023-02-22,1230,9
2996,dae3f348-c69f-4eae-bbe0-c5e393987c9b,Marcial Poza Valero S.L.N.E,Retail,China,Enterprise,2078,2024-08-20,2025-02-16,True,2025-01-13,87,40
2997,924354ec-155c-4c60-a10f-010dbcec6bbb,Reig y asociados S.L.,Finance,Mali,Starter,249,2025-12-05,2026-11-30,True,2026-01-28,1519,17
2998,c66d1228-e4f1-40f3-a248-9c67f3974add,Beltrán & Asociados S.A.,Health,Italia,Enterprise,3038,2023-03-03,2023-06-01,True,2023-03-24,3439,8


In [103]:
df_customers.to_csv('../data/raw/customers.csv', index=False)

## Generate Subscriptions
- sub_id
- customer_id
- plan
- billing_cycle
- start_date
- end_date
- discount_pct
- payment_method
- last_payment_status

In [ ]:
def generar_suscripcion(cliente: dict):
    if 12 > ((cliente["contract_end"] - cliente["contract_start"]).days / 30):
        billing_cycle = "Mensual"
    else: 
        billing_cycle = "Anual"

    r = random.randint(0,100)
    if cliente["churned"] == True:
        
        if r < 40:
            last_payment_status = "failed"
        elif 40 <= r and r <= 80:
            last_payment_status = "pending"
        else:
            last_payment_status = "success"
    else:
        if r < 70:
            last_payment_status = "success"
        elif 70 <= r and r <= 85:
            last_payment_status = "pending"
        else:
            last_payment_status = "failed"
    
    r = random.randint(0,100)
    if r < 5:
        discount_pct = 30
    elif 5 <= r and r <= 15:
        discount_pct = random.choice([20,30])
    else:
        discount_pct = 0

    return {
        "sub_id":               fake.uuid4(),
        "customer_id":          cliente["customer_id"],
        "plan":                 cliente["plan_type"],
        "billing_cycle":        billing_cycle,
        "start_date":           cliente["contract_start"],
        "end_date":             cliente["contract_end"],
        "discount_pct":         discount_pct,       
        "payment_method":       random.choice(["tarjeta de credito","transferencia bancaria","PayPal"]),      
        "last_payment_status":  last_payment_status
    }

In [149]:
subscriptions = [generar_suscripcion(cust) for cust in df_customers.to_dict('records')]

In [151]:
df_subscriptions = pd.DataFrame(subscriptions)
df_subscriptions

,sub_id,customer_id,plan,billing_cycle,start_date,end_date,discount_pct,payment_method,last_payment_status
0,5bfdb261-0de4-4381-8593-6b1e7dd2942e,949c8a0e-c8f2-40cc-8f97-527bcd35406a,Enterprise,Mensual,2023-12-19,2024-03-18,success,PayPal,pending
1,20ef23d8-ad86-42a5-87b6-4b489137925f,bb69a3c1-816c-454b-8d94-481186e8c99a,Starter,Anual,2025-07-26,2026-07-21,success,PayPal,success
2,79ddbdea-95c7-4a68-b464-07769d3f6ff3,5220a66c-36b8-4037-8c43-b08eb81203e2,Enterprise,Mensual,2024-02-23,2024-03-24,success,PayPal,success
3,1b3dddce-e76e-4648-85bc-29a6b23de5c3,b6878b37-e5da-460d-9d8b-aafa4eb5b029,Pro,Anual,2022-08-13,2023-08-08,success,PayPal,failed
4,9ce0b582-6682-4631-9b79-eada81a6c3a4,3439ad77-7bbb-41c9-88f8-1f276cf29283,Pro,Mensual,2023-01-31,2023-07-30,success,transferencia bancaria,success
...,...,...,...,...,...,...,...,...,...
2995,41d51b53-51cb-40bf-976e-3cb2b7721ad3,24d727ee-2ce0-4db9-a64f-e82458f8a86f,Enterprise,Mensual,2023-01-17,2023-07-16,success,tarjeta de credito,pending
2996,fb5e1001-e39f-4bb8-9e5a-748fee0c04a4,dae3f348-c69f-4eae-bbe0-c5e393987c9b,Enterprise,Mensual,2024-08-20,2025-02-16,success,PayPal,failed
2997,b0b7b843-f993-4440-9424-8a8d2f407641,924354ec-155c-4c60-a10f-010dbcec6bbb,Starter,Anual,2025-12-05,2026-11-30,success,tarjeta de credito,failed
2998,58ba2576-8cb0-4062-8b99-2bfac9b93095,c66d1228-e4f1-40f3-a248-9c67f3974add,Enterprise,Mensual,2023-03-03,2023-06-01,success,transferencia bancaria,pending


In [152]:
df_subscriptions.to_csv('../data/raw/subscriptions.csv', index=False)

## Usage Logs
- log_id
- customer_id
- user_id
- event_date
- feature_used
- session_duration_min
- login_count
- actions_count
- device_type

In [176]:
def generar_logs_cliente(cliente: dict):
    logs = []
    if cliente["churned"] == True: 
        n_logs = random.randint(10,80)
    else:
        n_logs = random.randint(50,100)

    if cliente["plan_type"] == "Starter":
        users = [fake.uuid4() for _ in range(np.random.randint(1,6))]
    elif cliente["plan_type"] == "Pro":
        users = [fake.uuid4() for _ in range(np.random.randint(5,21))]
    elif cliente["plan_type"] == "Enterprise":
        users = [fake.uuid4() for _ in range(np.random.randint(20,51))]

    modules = ["dashboard","task_manager","reports","integrations","billing","team_settings","calendar","notifications"]

    

    for _ in range(n_logs):
        if cliente["churned"] == True: 
            session_duration_min = random.randint(1,31)
            actions_count = random.randint(1,21)
        else:
            session_duration_min = random.randint(10,120)
            actions_count = random.randint(10,101)

        logs.append({
            "log_id":               fake.uuid4(),
            "customer_id":          cliente["customer_id"],
            "user_id":              random.choice(users),
            "event_date":           fake.date_between(start_date=cliente["contract_start"],end_date=cliente["contract_end"]),
            "feature_used":         random.choice(modules),
            "session_duration_min": session_duration_min,
            "login_count":          random.randint(1,5),
            "actions_count":        actions_count,
            "device_type":          random.choice(["desktop","mobile","tablet"])
        })

    return logs

In [177]:
logs = [generar_logs_cliente(customer) for customer in df_customers.to_dict('records')]